# Turbina NTNU con disco actuador (modelo A) en nibbler

Caso `wind_turbine_ansa_AD`: las condiciones del tutorial `Wind_Turbine` de HORSES3D (M = 0.02934, Re = 666666.67, LES Vreman), el rotor sustituido por un disco actuador con salto de presión uniforme, y p de 2 a 5 encadenados. Las decisiones y los números están en `wind_turbine_ansa_AD/metadata/LEEME.md`.

In [8]:
import os, glob, math, shutil
import pandas as pd
from FotR import GANDALF, FRODO

root = '/home/m.jaraiz/Documentos/COMAC/wind_turbine_ansa_AD'
sources = '/home/m.jaraiz/Documentos/COMAC/wind_turbine_ansa/sources'
utils = '/home/m.jaraiz/Documentos/utils_H3d'
repo = '~/repos/horses3d-gpu-ad'     # rama arr/actuator_disk (PR #121), compilada para GPU
case = os.path.join(root, 'outputs', 'ad_pressure_tsr_6.0')

## Δp, el parámetro de control

Δp está fijado en `ActuatorDef/Act_ActuatorDiskDef.dat`. Esta celda lo lee y calcula el empuje y el C_T que le corresponden, igual que el módulo: la fuerza de cada anillo es Δp·r_i·Δr_i·Δθ, con r_0 = radio del buje, y C_T se refiere a πR² y a la velocidad de referencia.

In [9]:
M, T_ref, p_ref, gamma, R_gas = 0.02934, 288.889, 101325.0, 1.4, 287.15
U = M * math.sqrt(gamma * R_gas * T_ref)
rho = p_ref / (R_gas * T_ref)
q = 0.5 * rho * U**2

lines = open(os.path.join(sources, 'ActuatorDef', 'Act_ActuatorDiskDef.dat')).read().splitlines()
value_after = lambda title: lines[next(i for i, l in enumerate(lines) if l.startswith(title)) + 1]
R = float(value_after('# Radius'))
hub = float(value_after('# Hub radius'))
first = next(i for i, l in enumerate(lines) if l.startswith('# Disk 1')) + 1
stations = [list(map(float, l.split())) for l in lines[first:] if l and not l.startswith('#')][:-1]
r = [hub] + [s[0] for s in stations]
dp = stations[0][1]
area = 2 * math.pi * sum(r[i] * (r[i] - r[i - 1]) for i in range(1, len(r)))
thrust = -dp * area
print(f'U = {U:.4f} m/s, rho = {rho:.5f} kg/m3, q = {q:.3f} Pa')
print(f'{len(stations)} estaciones, buje {hub}, R {R}, dp = {dp} Pa (uniforme: {len({s[1] for s in stations}) == 1})')
print(f'empuje = {thrust:.3f} N, C_T = {thrust / (q * math.pi * R**2):.4f}')

U = 9.9987 m/s, rho = 1.22145 kg/m3, q = 61.057 Pa
32 estaciones, buje 0.045, R 0.447, dp = -52.492568 Pa (uniforme: True)
empuje = 32.846 N, C_T = 0.8570


## Carpetas del caso

`overwrite=True` reescribe controles, malla, `ActuatorDef` y `run.sh` desde `sources`, pero nunca toca `RESULTS/` ni `LOG/`.

In [10]:
gdf = GANDALF(root_dir=root, ring='horses3d')
gdf.define_cases(method='external', external_dataframe=pd.DataFrame({'tsr': [6.0]}))
gdf.generate_folders(
    control_template=os.path.join(sources, 'wind_turbine_ad.control'),
    mesh_path=os.path.join(sources, 'Wind_turbine_disk_fluid_cgns_g2_mesh.h5'),
    tag='ad',
    folder_fmt='ad_pressure_tsr_{tsr:.1f}',
    p_levels=[2, 3, 4, 5],
    p_values={3: {'final time': 25.0}, 4: {'final time': 30.0}, 5: {'final time': 35.0}},
    extra_dirs=[os.path.join(sources, 'ActuatorDef')],
    run_script=os.path.join(utils, 'run.sh'),
    fsp=False,
    overwrite=True,
)

[horses3d ring] Inner zone(s) ['rotor_cylinder', 'disk_side'] have no '#define boundary' block in the template: added one with type = FreeSlipWall to every control. HORSES3D wants a type for every zone of the mesh, but these surfaces have no boundary faces (they stay interior), so the type is never applied.


,tsr,exist,folder
0,6.0,True,ad_pressure_tsr_6.0


El job 4353 (4 GPU) se canceló a los 300 pasos. Sus salidas se apartan, sin borrarlas, a `cancelled/job_4353/`: si `RESULTS/ad_g2p2.residuals` sigue ahí, `submit_cases` da p2 por empezado y no lo relanza.

In [11]:
aside = os.path.join(case, 'cancelled', 'job_4353')
old = glob.glob(os.path.join(case, 'RESULTS', 'ad_g2p2*')) + glob.glob(os.path.join(case, 'LOG', '*4353*'))
if old:
    os.makedirs(aside, exist_ok=True)
    for f in old:
        shutil.move(f, aside)
print(sorted(os.listdir(aside)) if os.path.isdir(aside) else 'nada que apartar')

['ad_g2p2.residuals', 'ad_g2p2_0000000000.hsol', 'ad_g2p2_Actuator_Disk_CP_CT_disk_001.dat', 'ad_g2p2_Actuator_Disk_Forces_disk_001.dat', 'ad_g2p2_Actuator_Disk_average_disk_001.dat', 'err4353.log', 'out4353.log']


## Recursos

Medidos el 22/11/2026 en n001 (H200 NVL) con 200 pasos (`wind_turbine_ansa_AD/benchmarks/`):

| s/paso | 1 GPU | 2 GPU | 4 GPU |
|---|---|---|---|
| p2 | 0.112 | 0.081 | 0.065 |
| p5 | 0.340 | 0.203 | 0.134 |

- **Por qué escala poco:** ninguna GPU pasa del 30 % de uso. El caso (38718 elementos) es pequeño para una H200, y el disco actuador añade unos 33–45 ms por paso: copia a la CPU el elemento de cada uno de sus 2304 puntos.
- **Pasos por orden:** p2 tiene unos 287000 pasos (dt = 6.96e-5 hasta t = 20); p3, p4 y p5 tienen 117000, 166000 y 217000.

| ngpu por orden | horas de reloj | GPU·h |
|---|---|---|
| todo 1 | 45 | 45 |
| **1, 1, 1, 2** | **37** | **49** |
| 1, 1, 2, 4 | 29 | 60 |
| todo 4 | 21 | 82 |

- **Elegido:** 1 GPU hasta p4 y 2 en p5. Las demás GPU quedan libres, por ejemplo para otro Δp en paralelo.
- **CPU por GPU:** con `cpus_per_gpu=4` basta, porque hay un proceso MPI por GPU y sin OpenMP.
- **Duración de cada trabajo:** ninguno pasa de unas 21 h (el límite es 48 h).

In [12]:
ngpu = {2: 4, 3: 4, 4: 2, 5: 2}#{2: 1, 3: 1, 4: 1, 5: 2}
launch = dict(arch='GPU', node='n001', ngpu=ngpu, cpus_per_gpu=16)

plan = gdf.submit_cases([2, 3, 4, 5], repo, **launch)
print(plan[['p', 'status', 'depends_on', 'command']].to_string())

   p   status depends_on                                                                                                                                                command
0  2  planned       None                          HORSES3D_REPO=/home/m.jaraiz/repos/horses3d-gpu-ad SOLVER=ns ./run.sh file_control_p2.control arr/actuator_disk GPU n001 4 16
1  3  planned       <p2>  HORSES3D_REPO=/home/m.jaraiz/repos/horses3d-gpu-ad SOLVER=ns DEPENDENCY=afterok:<p2> ./run.sh file_control_p3.control arr/actuator_disk GPU n001 4 16
2  4  planned       <p3>  HORSES3D_REPO=/home/m.jaraiz/repos/horses3d-gpu-ad SOLVER=ns DEPENDENCY=afterok:<p3> ./run.sh file_control_p4.control arr/actuator_disk GPU n001 2 16
3  5  planned       <p4>  HORSES3D_REPO=/home/m.jaraiz/repos/horses3d-gpu-ad SOLVER=ns DEPENDENCY=afterok:<p4> ./run.sh file_control_p5.control arr/actuator_disk GPU n001 2 16


Lanzamiento: p3, p4 y p5 esperan a que termine bien el anterior (`afterok`) y se cancelan si falla.

In [13]:
done = gdf.submit_cases([2, 3, 4, 5], repo, **launch, dry_run=False)
print(done[['p', 'status', 'job_id', 'depends_on', 'message']].to_string())

   p     status job_id depends_on message
0  2  submitted   4364       None        
1  3  submitted   4365       4364        
2  4  submitted   4366       4365        
3  5  submitted   4367       4366        


## Seguimiento

In [14]:
db = FRODO(root_dir=root, format='HORSES3D')
print(db.df_state[['case', 'p_available', 'p_finished', 'state', 'g']])

ct = os.path.join(case, 'RESULTS', 'ad_g2p2_Actuator_Disk_CP_CT_disk_001.dat')
if os.path.exists(ct):
    print(pd.read_csv(ct, sep=r'\s+').tail())

                  case   p_available p_finished    state  g
0  ad_pressure_tsr_6.0  [2, 3, 4, 5]         []  pending  2


/home/m.jaraiz/repos/fellowship-of-the-ring/FotR/characters/frodo.py:359: UserWarning: No Sets class for format 'HORSES3D'. Sets methods will not be available.
  warnings.warn(
/home/m.jaraiz/repos/fellowship-of-the-ring/FotR/characters/frodo.py:384: UserWarning: No Stats class for format 'HORSES3D'. Stats methods will not be available.
  warnings.warn(
